In [1]:
import pandas as pd

In [2]:
import os
os.chdir("/Users/Kashif/Library/CloudStorage/Box-Box/0.KB_NLP")

print("Working directory:", os.getcwd())

Working directory: /Users/Kashif/Library/CloudStorage/Box-Box/0.KB_NLP


In [3]:
df = pd.read_json(
    "data/raw/r_MenGetRapedToo_posts.jsonl",
    lines=True
)

In [4]:
df.shape
df.info()
df[["id", "title", "selftext", "is_self", "created_utc"]].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4463 entries, 0 to 4462
Columns: 135 entries, author to websocket_url
dtypes: bool(4), float64(65), int64(6), object(60)
memory usage: 4.5+ MB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4463 entries, 0 to 4462
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   id           4463 non-null   object
 1   title        4463 non-null   object
 2   selftext     4463 non-null   object
 3   is_self      4463 non-null   bool  
 4   created_utc  4463 non-null   int64 
dtypes: bool(1), int64(1), object(3)
memory usage: 144.0+ KB


In [5]:
# Standardize selftext temporarily for eligibility diagnostics
selftext_check = df["selftext"].astype("string").str.strip()

# Count common unusable selftext values
print("Total posts:", len(df))
print("Empty/whitespace:", (selftext_check == "").sum())
print("[deleted]:", selftext_check.str.lower().eq("[deleted]").sum())
print("[removed]:", selftext_check.str.lower().eq("[removed]").sum())

Total posts: 4463
Empty/whitespace: 401
[deleted]: 542
[removed]: 533


In [7]:
# Examine Reddit submission type (self/text post vs. link/media post)
# to determine whether is_self should be used as an eligibility criterion.
print(df["is_self"].value_counts(dropna=False))

# Define posts with usable body text.
# Excludes empty/whitespace-only bodies and Reddit placeholders for
# unavailable content ([deleted] and [removed]).
usable_selftext = ~(
    selftext_check.eq("")
    | selftext_check.str.lower().eq("[deleted]")
    | selftext_check.str.lower().eq("[removed]")
)

# Cross-tabulate submission type with availability of usable body text.
# Of 2,987 posts with usable selftext, 2,946 were self-posts and
# 41 were non-self (link/media) posts.
pd.crosstab(
    df["is_self"],
    usable_selftext,
    rownames=["is_self"],
    colnames=["usable_selftext"]
)

is_self
True     4013
False     450
Name: count, dtype: int64


usable_selftext,False,True
is_self,,
False,409,41
True,1067,2946


In [8]:
# Isolate the 41 non-self posts that nevertheless contain usable body text
# for manual inspection before deciding whether to exclude them.
nonself_with_text = df.loc[
    (~df["is_self"]) & usable_selftext,
    ["id", "title", "selftext", "is_self"]
].copy()

nonself_with_text.shape

# Inspect examples of non-self posts with available text.
nonself_with_text.head(10)

,id,title,selftext,is_self
1900,wbjezy,update to “i know the distance is hurting him”,i linked the original post there. to those who...,False
2185,10tbp9j,Is anyone noticing a trend?,let me know if you see something similar,False
2319,12jt1hz,My situation keeps getting worse,My rapist has 2 younger siblings a sister and ...,False
2347,132c3ab,I'm currently in the hospital due to a suicide...,"Text from my grandpa translated - ""Adam it's r...",False
2662,17n2fup,My story,So recently me and my wife started a podcast o...,False
2775,1amw7gw,The truth.,Link TW: Mentions of S/A - Just me sitting the...,False
3012,1dror9t,I was sent this picture by someone who was try...,Ok I'm a guy and I was sent this picture a wh...,False
3040,1e1xg0g,I keep falling into it,So like 15 days ago I made a posts explaining ...,False
3129,1f347v1,Trauma Recovery Video Series by Men in Healing...,Very well produced and insightful video series...,False
3130,1f34dqv,Not sure where to start.,"This is a little lengthy, for whatever reason ...",False


In [9]:
# Identify available Reddit metadata that can clarify why these posts
# were classified as non-self submissions.
[c for c in [
    "url",
    "domain",
    "post_hint",
    "is_reddit_media_domain",
    "is_video",
    "crosspost_parent",
    "permalink"
] if c in df.columns]

['url',
 'domain',
 'post_hint',
 'is_reddit_media_domain',
 'is_video',
 'crosspost_parent',
 'permalink']

In [10]:
# Reconstruct the diagnostic subset with link/media metadata.
# Inspection showed that the 41 records are link, image, gallery, video,
# and other multimedia submissions that can still contain user-authored text.
nonself_with_text = df.loc[
    (~df["is_self"]) & usable_selftext,
    [
        "id",
        "title",
        "selftext",
        "url",
        "domain",
        "post_hint",
        "is_reddit_media_domain",
        "is_video",
        "crosspost_parent"
    ]
].copy()

nonself_with_text[
    ["id", "title", "url", "domain", "post_hint", "is_video"]
].head(20)

,id,title,url,domain,post_hint,is_video
1900,wbjezy,update to “i know the distance is hurting him”,https://www.reddit.com/r/rapecounseling/commen...,reddit.com,NaN,0.0
2185,10tbp9j,Is anyone noticing a trend?,https://v.redd.it/2pe0x62qn6ga1,v.redd.it,hosted:video,1.0
2319,12jt1hz,My situation keeps getting worse,https://www.reddit.com/gallery/12jt1hz,reddit.com,NaN,0.0
2347,132c3ab,I'm currently in the hospital due to a suicide...,https://www.reddit.com/gallery/132c3ab,reddit.com,NaN,0.0
2662,17n2fup,My story,https://open.spotify.com/episode/4RAakl0STZBb3...,open.spotify.com,link,0.0
2775,1amw7gw,The truth.,https://youtu.be/-rIJpuXYdfw,youtu.be,rich:video,0.0
3012,1dror9t,I was sent this picture by someone who was try...,https://i.redd.it/tnv0h8psol9d1.jpeg,i.redd.it,image,0.0
3040,1e1xg0g,I keep falling into it,https://i.redd.it/lki26b1fn6cd1.png,i.redd.it,image,0.0
3129,1f347v1,Trauma Recovery Video Series by Men in Healing...,https://1in6.org/5-minutes-for-healing-trauma-...,1in6.org,NaN,0.0
3130,1f34dqv,Not sure where to start.,https://www.reddit.com/gallery/1f34dqv,reddit.com,NaN,0.0


# Assessing for Duplicated Observations

In [11]:
# Check whether the raw dataset contains duplicate Reddit submission IDs.
# Result: 0 duplicate IDs; all 4,463 records have unique submission IDs.
df["id"].duplicated().sum()

np.int64(0)

In [12]:
# Create a temporary working subset containing only posts with usable body text.
# This retains posts regardless of Reddit submission type (is_self) and excludes
# empty/whitespace-only, [deleted], and [removed] selftext.
# This is a diagnostic subset; the raw dataframe (df) remains unchanged.
df_usable = df.loc[usable_selftext].copy()

# Confirm the expected number of posts with usable body text.
df_usable.shape

(2987, 135)

**Exact duplicate assessment.** Following initial text-availability screening, posts with usable body text were examined for exact textual duplication. Duplicate Reddit IDs and duplicate textual content were assessed separately because unique submission IDs do not necessarily indicate unique textual observations. Potential duplicate text was identified for inspection prior to any exclusions to reduce the possibility of duplicated observations or train-test leakage.

In [13]:
# Count posts whose body text (selftext) exactly duplicates the body text
# of an earlier post, even if the Reddit submission IDs differ.
duplicate_selftext = df_usable["selftext"].duplicated().sum()

# Count posts for which BOTH title and body text exactly duplicate an
# earlier post. This is a more restrictive definition of exact duplication.
duplicate_title_selftext = df_usable.duplicated(
    subset=["title", "selftext"]
).sum()

print("Duplicate selftext:", duplicate_selftext)
print("Duplicate title + selftext:", duplicate_title_selftext)

Duplicate selftext: 11
Duplicate title + selftext: 5


**Inspection of exact textual duplicates.** Exact duplicate screening identified 11 posts whose body text duplicated another observation, including 5 posts with identical title and body text. Because duplicate text may reflect reposts, updates, crossposts, boilerplate text, or other legitimate patterns, duplicate observations were manually inspected before determining whether deduplication was warranted.

In [14]:
# Identify ALL posts belonging to a duplicated-selftext group.
# keep=False retains both the first occurrence and subsequent duplicates,
# allowing the complete duplicate groups to be inspected.
duplicate_text_posts = df_usable.loc[
    df_usable["selftext"].duplicated(keep=False),
    ["id", "title", "selftext", "created_utc", "is_self"]
].copy()

# Sort identical body text together and order posts chronologically
# within each duplicate group.
duplicate_text_posts = duplicate_text_posts.sort_values(
    ["selftext", "created_utc"]
)

# Number of posts involved in duplicate-text groups.
duplicate_text_posts.shape

(19, 5)

In [15]:
# Display the duplicate-text observations for manual inspection.
duplicate_text_posts[
    ["id", "title", "selftext", "created_utc", "is_self"]
]

,id,title,selftext,created_utc,is_self
342,9nebke,"The ""Good Rape Crisis Center Guide""","As many users here will know, rape crisis cent...",1539294682,True
443,bfinj1,"The ""Good Rape Crisis Center Guide""","As many users here will know, rape crisis cent...",1555803907,True
988,k2dhvw,"The ""Good Rape Crisis Center Guide""","As many users here will know, rape crisis cent...",1606524416,True
1351,oaze0n,"The ""Good Rape Crisis Center Guide""—updated","As many users here will know, rape crisis cent...",1625066092,True
2747,199i0jv,Survivors help,"Hi, I'm 15 years old and a victim of statutory...",1705553736,True
2749,199klco,Help.,"Hi, I'm 15 years old and a victim of statutory...",1705563023,True
3463,1j7roj7,Had a heart attack,"I just had a heart attack, they don't know why...",1741585127,True
3464,1j7row5,Getting a heart catheter,"I just had a heart attack, they don't know why...",1741585169,True
2836,1blmk5w,Can rape turns me gay?,Im 21 and straight. 3 weeks ago I went to a pa...,1711179279,True
2837,1blmk67,Can rape turns me gay?,Im 21 and straight. 3 weeks ago I went to a pa...,1711179281,True


**Characterization of duplicate groups.** Nineteen posts belonged to groups containing identical body text. These included both near-simultaneous duplicate submissions and text reposted at later dates. Because the analytic unit is the textual Reddit post, duplicate groups were characterized before determining whether repeated textual observations should be retained, excluded, or grouped during subsequent sampling and model development.

In [16]:
# Assign a group number to each unique duplicated selftext value.
# Posts with identical body text receive the same duplicate_group value.
duplicate_text_posts["duplicate_group"] = (
    duplicate_text_posts.groupby("selftext", sort=False).ngroup() + 1
)

# Convert Reddit Unix timestamps to readable UTC dates to help distinguish
# near-simultaneous duplicate submissions from later reposts.
duplicate_text_posts["created_datetime"] = pd.to_datetime(
    duplicate_text_posts["created_utc"],
    unit="s",
    utc=True
)

# Display duplicate groups without printing the full body text.
duplicate_text_posts[
    ["duplicate_group", "id", "title", "created_datetime", "is_self"]
].sort_values(["duplicate_group", "created_datetime"])

,duplicate_group,id,title,created_datetime,is_self
342,1,9nebke,"The ""Good Rape Crisis Center Guide""",2018-10-11 21:51:22+00:00,True
443,1,bfinj1,"The ""Good Rape Crisis Center Guide""",2019-04-20 23:45:07+00:00,True
988,1,k2dhvw,"The ""Good Rape Crisis Center Guide""",2020-11-28 00:46:56+00:00,True
1351,1,oaze0n,"The ""Good Rape Crisis Center Guide""—updated",2021-06-30 15:14:52+00:00,True
2747,2,199i0jv,Survivors help,2024-01-18 04:55:36+00:00,True
2749,2,199klco,Help.,2024-01-18 07:30:23+00:00,True
3463,3,1j7roj7,Had a heart attack,2025-03-10 05:38:47+00:00,True
3464,3,1j7row5,Getting a heart catheter,2025-03-10 05:39:29+00:00,True
2836,4,1blmk5w,Can rape turns me gay?,2024-03-23 07:34:39+00:00,True
2837,4,1blmk67,Can rape turns me gay?,2024-03-23 07:34:41+00:00,True


**Duplicate-handling decision.** Eight groups comprising 19 posts contained identical body text. Duplicate patterns included near-simultaneous submissions as well as identical content reposted at later dates. Because the objective of the corpus is to develop and validate a classifier of textual content rather than to examine posting frequency or reposting behavior, exact duplicate body text was treated as redundant for model development. One observation per unique body text was retained, using the earliest submission based on created_utc. This prevents identical textual observations from being overrepresented or appearing across development and test samples. The original raw dataset remains unchanged.

In [17]:
# Create a candidate analytic corpus from posts with usable body text.
# Sort chronologically so that, when identical selftext occurs more than once,
# the earliest Reddit submission is retained.
df_candidate = (
    df_usable
    .sort_values("created_utc")
    .drop_duplicates(subset=["selftext"], keep="first")
    .copy()
)

# Confirm the number of observations after exact-text deduplication.
print("Posts before deduplication:", len(df_usable))
print("Posts after deduplication:", len(df_candidate))
print("Posts removed as exact selftext duplicates:",
      len(df_usable) - len(df_candidate))

Posts before deduplication: 2987
Posts after deduplication: 2976
Posts removed as exact selftext duplicates: 11


In [18]:
# Verify that Reddit IDs remain unique and that no exact duplicate
# selftext remains in the candidate corpus.
print("Duplicate IDs remaining:",
      df_candidate["id"].duplicated().sum())

print("Duplicate selftext remaining:",
      df_candidate["selftext"].duplicated().sum())

Duplicate IDs remaining: 0
Duplicate selftext remaining: 0


**Exact-text deduplication.** After restricting the corpus to posts with usable body text (n = 2,987), exact duplicate `selftext` was removed while retaining the earliest submission within each duplicate-text group. Eleven redundant observations were removed, resulting in 2,976 candidate posts. The resulting corpus contained no duplicate Reddit IDs or exact duplicate body text.

In [19]:
import re

# Create a normalized version of selftext ONLY for duplicate detection.
# The original selftext is preserved unchanged for subsequent NLP analyses.
def normalize_for_duplicate_check(text):
    text = str(text).lower()
    text = re.sub(r"\s+", " ", text).strip()
    return text

df_candidate["selftext_dupcheck"] = (
    df_candidate["selftext"]
    .apply(normalize_for_duplicate_check)
)

# Count texts that become identical after normalization.
df_candidate["selftext_dupcheck"].duplicated().sum()

np.int64(4)

**Inspection of normalized-text duplicates.** Four additional observations were identified as duplicate occurrences after normalizing capitalization and whitespace. All observations belonging to these normalized duplicate groups were retained for manual inspection before determining whether they represented substantively identical textual content.

In [20]:
# Identify ALL observations belonging to a normalized duplicate-text group.
# keep=False includes both the first occurrence and subsequent duplicate(s).
normalized_duplicate_posts = df_candidate.loc[
    df_candidate["selftext_dupcheck"].duplicated(keep=False),
    [
        "id",
        "title",
        "selftext",
        "selftext_dupcheck",
        "created_utc",
        "is_self"
    ]
].copy()

# Convert timestamps to readable UTC dates for inspection.
normalized_duplicate_posts["created_datetime"] = pd.to_datetime(
    normalized_duplicate_posts["created_utc"],
    unit="s",
    utc=True
)

# Assign a group number to posts with the same normalized text.
normalized_duplicate_posts["duplicate_group"] = (
    normalized_duplicate_posts
    .groupby("selftext_dupcheck", sort=False)
    .ngroup() + 1
)

# Display the duplicate groups without the long text fields.
normalized_duplicate_posts[
    ["duplicate_group", "id", "title", "created_datetime", "is_self"]
].sort_values(["duplicate_group", "created_datetime"])

,duplicate_group,id,title,created_datetime,is_self
2809,1,1b375o7,I was blackmailed into doing sexual stuff at y...,2024-02-29 18:17:14+00:00,True
2905,1,1cg3t00,I was blackmailed into doing sexual stuff at y...,2024-04-29 16:43:04+00:00,True
4029,2,1s65lxg,I need someone to talk to,2026-03-28 16:54:31+00:00,True
4035,2,1s904sw,"I need help , please someone talk to me",2026-03-31 21:05:19+00:00,True
4420,3,1vpfhkx,My dad raped me when I was younger. I have so ...,2026-08-15 22:04:05+00:00,True
4421,3,1vpfi1e,My dad raped me when I was younger. I have so ...,2026-08-15 22:04:41+00:00,True
4428,4,1vpqe1u,Unless took advantage,2026-08-16 07:17:01+00:00,True
4429,4,1vpqem4,Uncle took advantage,2026-08-16 07:17:58+00:00,True


**Normalized-text deduplication.** After exact-text deduplication, four additional duplicate occurrences were identified after normalization of capitalization and whitespace. These represented four pairs of posts with substantively identical body text, including both near-simultaneous duplicate submissions and later reposts. Because normalization altered only capitalization and whitespace rather than lexical content, these observations were treated as redundant textual inputs. The earliest submission within each normalized duplicate group was retained.

In [21]:
# Sort chronologically so the earliest occurrence of each normalized
# body text is retained.
df_candidate = (
    df_candidate
    .sort_values("created_utc")
    .drop_duplicates(subset=["selftext_dupcheck"], keep="first")
    .copy()
)

# Confirm the corpus size after normalized-text deduplication.
print("Posts after normalized-text deduplication:", len(df_candidate))
print(
    "Normalized duplicate selftext remaining:",
    df_candidate["selftext_dupcheck"].duplicated().sum()
)

Posts after normalized-text deduplication: 2972
Normalized duplicate selftext remaining: 0


# Construction of model text. 

Following eligibility screening and conservative deduplication, the post title and body were preserved as separate fields and combined into a single text field for subsequent NLP analyses. No linguistic preprocessing (e.g., stop-word removal, stemming, lemmatization, or punctuation removal) was applied at this stage so that the original linguistic and contextual information remained available for model-specific preprocessing.

In [22]:
# Preserve the original title and body text in explicitly named fields.
# These fields will remain unchanged throughout subsequent preprocessing.
df_candidate["title_raw"] = df_candidate["title"].astype("string")
df_candidate["selftext_raw"] = df_candidate["selftext"].astype("string")

# Construct a single document-level text field containing the post title
# followed by the body. A newline separator preserves the conceptual
# boundary between the two components without altering their content.
df_candidate["text_raw"] = (
    df_candidate["title_raw"].str.strip()
    + "\n\n"
    + df_candidate["selftext_raw"].str.strip()
)

# Inspect the resulting fields.
df_candidate[
    ["id", "title_raw", "selftext_raw", "text_raw"]
].head()

,id,title_raw,selftext_raw,text_raw
2,1qldir,men don't get raped,lol let's be real here,men don't get raped lol let's be real here
3,20rz6j,Looking for resources in helping my husband de...,As the title says I'm looking for resources to...,Looking for resources in helping my husband de...
4,217jvr,Crossposting this from r/rape,When I was a kid from about 9-15 my mother wou...,Crossposting this from r/rape When I was a ki...
7,2hojbd,hello,i am an arab. watch out im going to rape you!!...,hello i am an arab. watch out im going to rap...
8,2ic942,Did I rape him?,My boyfriend (M/26) and I (F/24) were in bed t...,Did I rape him? My boyfriend (M/26) and I (F/...


In [23]:
# Verify that every candidate post has non-empty combined model text.
print(
    "Empty combined texts:",
    df_candidate["text_raw"].str.strip().eq("").sum()
)

print("Candidate corpus size:", len(df_candidate))

Empty combined texts: 0
Candidate corpus size: 2972


# Document Length Diagnostics

**Document-length characterization.** Document length was characterized prior to model development to inform subsequent tokenization and model-selection decisions. Character and whitespace-delimited word counts were calculated using the combined title and body text. These descriptive measures were used to characterize the distribution of document lengths and identify unusually short or long observations; they were not used as model inputs.

In [24]:
# Calculate basic document-length measures from the combined raw text.
# These variables are descriptive only and do not modify the text itself.
df_candidate["char_count"] = df_candidate["text_raw"].str.len()

df_candidate["word_count"] = (
    df_candidate["text_raw"]
    .str.split()
    .str.len()
)

# Summarize the distribution of document lengths.
df_candidate[["char_count", "word_count"]].describe(
    percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
).round(1)

,char_count,word_count
count,2972.0,2972.0
mean,1789.4,345.6
std,2066.3,397.5
min,18.0,4.0
25%,580.8,113.0
50%,1199.5,231.0
75%,2226.0,430.2
90%,3905.9,759.8
95%,5281.1,1017.3
99%,10372.8,2003.6


In [25]:
# Inspect the shortest posts for obvious non-substantive content
# or data artifacts that may have passed the initial availability screen.
df_candidate.nsmallest(10, "word_count")[
    ["id", "title_raw", "selftext_raw", "word_count"]
]

,id,title_raw,selftext_raw,word_count
3785,1otr7v7,Hi,Hi everyone 👋.,4
2572,16a78zq,ZKSyncSwap: Swaps That Airdrop,https://zsyncswap.technology/,5
119,64275i,very true?,mem dont rape it,6
3750,1oduxza,French-speaking subreddit for male victims,https://www.reddit.com/r/HommesVictimesVS/,6
1109,lea714,What was your rapist,[removed] [View Poll](https://www.reddit.com/...,7
1674,sgo6z8,Guys I having a ptsd attack,Please help,8
2101,zo5o6g,Any trans men in here?,[removed] [View Poll](https://www.reddit.com/...,8
2877,1c3u5ae,Would you rape a young boy??,Just curious,8
4318,1up184t,Gay no limit session,052bcf3cfde1a70d16340f9eb44dcec292225c381e6815...,8
2,1qldir,men don't get raped,lol let's be real here,9


In [26]:
# Inspect the longest posts for obvious scraping or formatting artifacts.
df_candidate.nlargest(10, "word_count")[
    ["id", "title_raw", "word_count"]
]

,id,title_raw,word_count
2797,1aystv9,"I know it wasnt even SA, so is it weird that i...",4594
385,ah8x4l,I [24M] am a survivor of domestic violence and...,4017
1168,m00g8n,i was raped by a roommate when i was 18. i am ...,3859
1304,np2tr4,"Taken advantage of many times when drunk, lack...",3789
1692,stqppu,How Can I Help Him?,3739
759,gvaeq0,For Disguised as Friend,3495
125,6bk1f4,My Story,3260
3184,1fxx89e,My Story of 13 Years,3170
327,9eglki,Ever had authorities laugh at you? “She just l...,3131
1391,ot2p7e,"Sexually Assaulted by a boss 4 years ago, fina...",3121


**Inspection of document-length extremes.** Document lengths were strongly right-skewed, with a median of 231 words and a maximum of 4,594 words. Manual inspection of the shortest and longest observations did not support imposing an arbitrary minimum or maximum document length. Short posts may contain substantively relevant text, while long posts may represent legitimate survivor narratives. Inspection did, however, identify posts in which Reddit-generated placeholders such as `[removed]` were followed by residual interface text (e.g., poll links), requiring refinement of the text-availability criterion.

In [27]:
# Identify candidate posts whose body begins with a Reddit unavailable-content
# placeholder but was not caught by the earlier exact-placeholder screen.
placeholder_remnants = df_candidate.loc[
    df_candidate["selftext_raw"]
    .str.strip()
    .str.lower()
    .str.match(r"^\[(removed|deleted)\]"),
    ["id", "title_raw", "selftext_raw", "word_count"]
].copy()

print("Placeholder-remnant posts:", len(placeholder_remnants))

placeholder_remnants

Placeholder-remnant posts: 2


,id,title_raw,selftext_raw,word_count
1109,lea714,What was your rapist,[removed] [View Poll](https://www.reddit.com/...,7
2101,zo5o6g,Any trans men in here?,[removed] [View Poll](https://www.reddit.com/...,8


**Refinement of unavailable-text exclusions.** Inspection identified two posts in which the original body text had been removed but residual Reddit-generated poll text remained after the `[removed]` placeholder. Because these observations did not contain the original user-authored body text, they were excluded under the same unavailable-content criterion as posts containing only `[removed]` or `[deleted]`. No minimum or maximum document-length criterion was imposed.

In [28]:
# Exclude posts beginning with Reddit unavailable-content placeholders
# ([removed] or [deleted]), including cases followed by residual interface
# text such as Reddit poll links.
placeholder_mask = (
    df_candidate["selftext_raw"]
    .str.strip()
    .str.lower()
    .str.match(r"^\[(removed|deleted)\]")
)

df_candidate = df_candidate.loc[~placeholder_mask].copy()

# Confirm the resulting candidate corpus size.
print("Candidate corpus size:", len(df_candidate))

Candidate corpus size: 2970


In [29]:
# Final verification of the core corpus-cleaning criteria.
print("Duplicate IDs:",
      df_candidate["id"].duplicated().sum())

print("Normalized duplicate bodies:",
      df_candidate["selftext_dupcheck"].duplicated().sum())

print("Unavailable-content placeholders:",
      df_candidate["selftext_raw"]
      .str.strip()
      .str.lower()
      .str.match(r"^\[(removed|deleted)\]")
      .sum())

print("Empty combined texts:",
      df_candidate["text_raw"].str.strip().eq("").sum())

Duplicate IDs: 0
Normalized duplicate bodies: 0
Unavailable-content placeholders: 0
Empty combined texts: 0


In [30]:
print("Rows entering language screening:", len(df_candidate))

Rows entering language screening: 2970


**Final corpus-cleaning verification.** Following text-availability screening, duplicate removal, and exclusion of residual unavailable-content artifacts, the candidate corpus contained 2,970 posts. All retained observations had unique Reddit submission IDs, unique body text after conservative capitalization/whitespace normalization, available user-authored body text, and non-empty combined title-and-body text. No minimum or maximum document-length criterion was imposed. The resulting corpus was carried forward for model-specific text characterization and subsequent annotation and classification procedures.

# Language Eligibility Screening

**Language eligibility assessment.** Because subsequent classifier development will use English-language pretrained models, the candidate corpus was screened for non-English submissions prior to annotation sampling. Automated language identification was used as an initial screening procedure rather than as a definitive exclusion criterion. Posts flagged as potentially non-English were manually reviewed to distinguish genuinely non-English submissions from language-detection errors.

In [ ]:
# Install the language-identification package used to screen the
# Candidate Corpus for potentially non-English submissions.
%pip install lingua-language-detector

In [31]:
# Initialize the language detector using all languages supported by Lingua.
# Automated detection is used only to identify posts requiring manual review.
from lingua import LanguageDetectorBuilder

detector = LanguageDetectorBuilder.from_all_languages().build()

# Detect the language of each complete title + body document.
df_candidate["language"] = df_candidate["text_raw"].apply(
    lambda text: detector.detect_language_of(str(text))
)

# Convert Lingua language objects to readable language names.
df_candidate["language"] = df_candidate["language"].apply(
    lambda lang: lang.name if lang is not None else "UNKNOWN"
)

# Summarize the automated language classifications.
language_counts = df_candidate["language"].value_counts()

print(language_counts)

language
ENGLISH       2958
FRENCH           5
PORTUGUESE       3
SPANISH          2
YORUBA           1
WELSH            1
Name: count, dtype: int64


In [32]:
# Identify posts manually confirmed as non-English.
# Manual review showed that all French, Portuguese, and Spanish detections
# were accurate, while the Yoruba and Welsh detections were false positives.
confirmed_non_english = df_candidate.loc[
    df_candidate["language"].isin(
        ["FRENCH", "PORTUGUESE", "SPANISH"]
    )
].copy()

print("Confirmed non-English posts:", len(confirmed_non_english))

# Verify the expected result before excluding any observations.
assert len(confirmed_non_english) == 10

Confirmed non-English posts: 10


### Manual Review of Flagged Posts

Automated language identification flagged 12 submissions as potentially non-English. Manual review confirmed that 10 submissions were primarily written in a language other than English (5 French, 3 Portuguese, and 2 Spanish). Two submissions classified by the detector as Yoruba and Welsh were determined upon manual review to be written in English and were retained.

The 10 confirmed non-English submissions were excluded because subsequent classifier development is restricted to English-language text. This reduced the Candidate Corpus from 2,970 to 2,960 submissions.

In [33]:
# Exclude the 10 submissions manually confirmed as non-English.
df_candidate = df_candidate.loc[
    ~df_candidate["id"].isin(confirmed_non_english["id"])
].copy()

# Verify the final English-language Candidate Corpus.
print("Final Candidate Corpus:", df_candidate.shape)
print("Unique IDs:", df_candidate["id"].nunique())
print(
    "Confirmed non-English posts remaining:",
    df_candidate["id"].isin(confirmed_non_english["id"]).sum()
)

# Safety checks before proceeding.
assert len(df_candidate) == 2960
assert df_candidate["id"].nunique() == 2960

Final Candidate Corpus: (2960, 142)
Unique IDs: 2960
Confirmed non-English posts remaining: 0


# Transformer Token-Length Diagnostics

Because transformer input limits are defined in tokens rather than words, document lengths were additionally characterized using a pretrained tokenizer representative of a standard encoder architecture. Tokenization was performed solely to characterize sequence lengths and did not constitute model training or fitting. These diagnostics were used to inform subsequent decisions regarding truncation, segmentation, or long-context architectures.

In [ ]:
# Load a pretrained RoBERTa tokenizer for document-length diagnostics.
# No model is being trained or fine-tuned at this stage.
# Install Hugging Face Transformers for pretrained tokenizers/models.
%pip install transformers

In [34]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("roberta-base")

[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [36]:
# Tokenize each combined title + body document WITHOUT truncation.
# We retain only the token counts; these tokenized sequences are not
# being used for model training at this stage.
df_candidate["roberta_token_count"] = df_candidate["text_raw"].apply(
    lambda text: len(
        tokenizer(
            text,
            add_special_tokens=True,
            truncation=False
        )["input_ids"]
    )
)

# Summarize the token-length distribution.
df_candidate["roberta_token_count"].describe(
    percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
).round(1)

count    2960.0
mean      429.4
std       492.0
min        10.0
25%       143.8
50%       287.5
75%       533.0
90%       944.0
95%      1249.3
99%      2477.4
max      5954.0
Name: roberta_token_count, dtype: float64

In [37]:
# Quantify how many documents exceed common transformer context lengths.
# This helps determine whether truncation or a long-document strategy
# needs to be considered during model development.
for threshold in [512, 1024, 2048, 4096]:
    n_over = (df_candidate["roberta_token_count"] > threshold).sum()
    pct_over = n_over / len(df_candidate) * 100

    print(
        f">{threshold} tokens: {n_over} posts "
        f"({pct_over:.1f}%)"
    )

>512 tokens: 779 posts (26.3%)
>1024 tokens: 246 posts (8.3%)
>2048 tokens: 46 posts (1.6%)
>4096 tokens: 7 posts (0.2%)


**Transformer token-length diagnostics.** Using the RoBERTa-base tokenizer, documents contained a median of 288 tokens (IQR: 144–533), with a mean of 429 tokens. Of the 2,960 candidate documents, 779 (26.3%) exceeded 512 tokens, 246 (8.3%) exceeded 1,024 tokens, 46 (1.6%) exceeded 2,048 tokens, and 7 (0.2%) exceeded 4,096 tokens. These findings indicate that straightforward 512-token truncation would affect a substantial minority of posts and support explicit evaluation of long-document handling strategies during model development.

# Candidate Corpus Freeze

Following eligibility screening, conservative deduplication, text construction, and document-length characterization, the candidate corpus was frozen prior to annotation sampling. Temporary variables used exclusively for duplicate detection were removed from the saved analytic dataset, while original text and descriptive document-length variables were retained.

In [38]:
%pip install pyarrow

Note: you may need to restart the kernel to use updated packages.


In [40]:
# Freeze the finalized Candidate Corpus for downstream analysis.
# Temporary variables used only for duplicate detection are removed,
# while the original Reddit metadata and derived analytic variables are retained.
df_frozen = df_candidate.drop(
    columns=["selftext_dupcheck"]
).copy()

print("Frozen corpus shape:", df_frozen.shape)

Frozen corpus shape: (2960, 142)


In [41]:
# Examine how much poll metadata exists in the candidate corpus.
print("Non-missing poll_data:", df_frozen["poll_data"].notna().sum())

df_frozen.loc[
    df_frozen["poll_data"].notna(),
    ["id", "title_raw", "poll_data"]
].head(10)

Non-missing poll_data: 47


,id,title_raw,poll_data
2411,145mbl8,Was my friend SA’ed?,"{'is_prediction': False, 'options': [{'id': '2..."
3677,1neucda,Is anyone else disgusted by sex?,False
3679,1nexf6v,Im intensely desiring to be in pain as a woman...,False
3682,1nfwix7,Am I wrong for questioning the story of my gir...,False
3683,1nh6bnd,HELPP HELPP BIG WOMEN RAPING ME,False
3712,1nw7i22,Targeted in California,False
3713,1ny2weg,Targeted,False
3728,1o3mbvj,Women don’t accept men with trauma,False
3734,1o6nz3t,Given viagra,False
3742,1oat8tb,The world doesn’t care if men or boys are rape...,False


In [42]:
# Remove Reddit poll metadata from the processed corpus.
# Poll mechanics are not part of the planned text analyses and remain
# preserved in the immutable raw JSONL if needed in the future.
df_frozen = df_frozen.drop(columns=["poll_data"])

print("Frozen corpus shape:", df_frozen.shape)

Frozen corpus shape: (2960, 141)


# Save the Finalized Corpus

In [43]:
# Save the finalized Candidate Corpus as JSON Lines.
# JSONL preserves the broad Reddit metadata structure while remaining
# convenient for downstream processing in Python.
df_frozen.to_json(
    "data/processed/reddit_candidate_corpus.jsonl",
    orient="records",
    lines=True,
    force_ascii=False
)

print("Saved Candidate Corpus.")

Saved Candidate Corpus.


In [44]:
# Reload the saved Candidate Corpus to verify that the export preserved
# the expected number of observations and unique submission IDs.
df_check = pd.read_json(
    "data/processed/reddit_candidate_corpus.jsonl",
    lines=True
)

print("Saved corpus shape:", df_check.shape)
print("IDs unique:", df_check["id"].is_unique)
print("Missing IDs:", df_check["id"].isna().sum())

Saved corpus shape: (2960, 141)
IDs unique: True
Missing IDs: 0
